In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin1")
df["Order Date"] = pd.to_datetime(df["Order Date"])

monthly = (
    df.set_index("Order Date")["Sales"]
    .resample("MS").sum()
    .to_frame("Sales")
)
print(monthly.shape)
monthly.tail()

(48, 1)


,Sales
Order Date,
2017-08-01,63120.8880
2017-09-01,87866.6520
2017-10-01,77776.9232
2017-11-01,118447.8250
2017-12-01,83829.3188


In [2]:
data = monthly.copy()
data["lag_1"] = data["Sales"].shift(1)
data["lag_2"] = data["Sales"].shift(2)
data["lag_3"] = data["Sales"].shift(3)
data["lag_12"] = data["Sales"].shift(12)
data["roll_3"] = data["Sales"].shift(1).rolling(3).mean()
data["month"] = data.index.month
data = data.dropna()

test_size = 6
train, test = data.iloc[:-test_size], data.iloc[-test_size:]

features = ["lag_1", "lag_2", "lag_3", "lag_12", "roll_3", "month"]
X_train, y_train = train[features], train["Sales"]
X_test, y_test = test[features], test["Sales"]
print(len(train), len(test))

30 6


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_pred = test["roll_3"]

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:15s} MAE={mae:9,.0f}  RMSE={rmse:9,.0f}  MAPE={mape:5.1f}%")

report("Baseline", y_test, baseline_pred)
report("Random Forest", y_test, rf_pred)

Baseline        MAE=   19,299  RMSE=   24,003  MAPE= 21.6%
Random Forest   MAE=   15,874  RMSE=   18,588  MAPE= 19.2%


In [4]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=data.index, y=data["Sales"], name="Actual")
fig.add_scatter(x=test.index, y=rf_pred, name="Random Forest")
fig.add_scatter(x=test.index, y=baseline_pred, name="Baseline")
fig.update_layout(title="Forecast vs Actual (last 6 months)")
fig.show()